In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "mistralai/Mistral-7B-v0.3"  # or your local path

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

print("✅ Model loaded successfully!")


tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\Users\mahes\anaconda3\envs\Mahii\Lib\site-packages\huggingface_hub\file_download.py:120: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mahes\.cache\huggingface\hub\models--mistralai--Mistral-7B-v0.3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HT

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


✅ Model loaded successfully!


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "mistralai/Mistral-7B-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",  # uses GPU if available
    torch_dtype="auto"  # automatically chooses float16 if possible
)

print("✅ Model loaded successfully!")


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.


✅ Model loaded successfully!


In [2]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

prompt = "In a distant galaxy, humans discovered"
output = generator(prompt, max_new_tokens=50, do_sample=True, temperature=0.7)
print(output[0]['generated_text'])

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In a distant galaxy, humans discovered a planet populated by aliens of various species – the inhabitants of the planet. Their way of life is based on a very strict hierarchy, where each inhabitant has a certain position. The lowest rank is commoner, and the highest is the


In [ ]:
import pandas as pd
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
from time import sleep

# ===============================
# Load Mistral 7B
# ===============================
model_name = "mistralai/Mistral-7B-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
)

# ===============================
# Rule-based feature extractor (fast)
# ===============================
def extract_rule_based_features_fast(text):
    features = {}
    
    # Numeric
    value_match = re.search(r"Value:\s*([\d.]+)", text)
    features['weight_value'] = float(value_match.group(1)) if value_match else 0
    
    pack_match = re.search(r"Pack of (\d+)", text, re.IGNORECASE)
    features['pack_size'] = int(pack_match.group(1)) if pack_match else 1
    
    total_items_match = re.search(r"Includes .*? (\d+) (cookies|items|units)", text, re.IGNORECASE)
    features['total_items'] = int(total_items_match.group(1)) if total_items_match else features['pack_size']
    features['items_per_pack'] = features['total_items'] / (features['pack_size'] or 1)
    
    # Bullet points
    bullets = re.findall(r"Bullet Point \d+:\s*(.*)", text)
    features['num_bullets'] = len(bullets)
    
    all_text = text.replace("\n", " ").lower()
    features['description_length_words'] = len(all_text.split())
    features['description_length_chars'] = len(all_text)
    features['num_sentences'] = len(re.findall(r"[.!?]", all_text))
    features['num_numbers'] = len(re.findall(r"\d+", all_text))
    features['avg_word_length'] = sum(len(w) for w in all_text.split()) / (len(all_text.split()) or 1)
    
    # Boolean / categorical
    features['is_gluten_free'] = int("gluten-free" in all_text)
    features['is_nut_free'] = int("nut-free" in all_text)
    features['is_organic'] = int("organic" in all_text or "org" in all_text)
    features['is_wine'] = int("wine" in all_text or "alcohol" in all_text or "wine kit" in all_text)
    features['is_spice_seasoning'] = int("seasoning" in all_text or "spice" in all_text)
    features['is_liquid'] = int(re.search(r"(fl oz|l\b)", all_text) is not None)
    features['is_powder'] = int("powder" in all_text)
    features['contains_alcohol'] = features['is_wine']
    features['contains_spices'] = features['is_spice_seasoning']
    features['contains_ingredients'] = int("ingredients:" in all_text)
    
    # Flavors
    flavors = ["butter", "blue cheese", "basil", "cheddar", "vanilla"]
    for f in flavors:
        features[f'has_flavor_{f.replace(" ", "_")}'] = int(f in all_text)
    
    # Occasions
    occasions = ["birthday", "wedding", "anniversary", "party", "gift"]
    for occ in occasions:
        features[f'is_{occ}'] = int(occ in all_text)
    
    # Marketing keywords
    marketing_keywords = ["classic", "premium", "delicious", "trusted", "rich", "fun", "versatile"]
    features['num_marketing_keywords'] = sum(all_text.count(k) for k in marketing_keywords)
    
    # Derived
    features['num_occasions'] = sum(features[f'is_{occ}'] for occ in occasions)
    features['text_complexity'] = features['description_length_words'] / (features['num_sentences'] or 1)
    features['volume_per_unit'] = features['weight_value'] / (features['pack_size'] or 1)
    
    return features

# ===============================
# LLM enrichment (batched)
# ===============================
def enrich_features_batch(text_list):
    batch_results = []
    for text in text_list:
        prompt = f"""Extract in JSON format:
- product_type (cookie, wine, spice, cereal, etc.)
- implicit giftable (1 or 0)
- flavor_profile (list)
Product Text: {text}"""
        
        try:
            output = generator(prompt, max_new_tokens=150)[0]['generated_text']
            json_start = output.find("{")
            json_end = output.rfind("}") + 1
            llm_features = eval(output[json_start:json_end])
        except:
            llm_features = {
                "llm_product_type": None,
                "llm_is_giftable": None,
                "llm_flavors": None
            }
        batch_results.append(llm_features)
        sleep(0.05)
    return batch_results

# ===============================
# Process dataset
# ===============================
def process_dataset_optimized(input_csv, output_csv, batch_size=200):
    df = pd.read_csv(input_csv)
    if 'sample_id' not in df.columns:
        df['sample_id'] = df.index
    
    print("➡ Extracting rule-based features...")
    rule_features = []
    for idx, row in df.iterrows():
        feats = extract_rule_based_features_fast(row['catalog_content'])
        feats['sample_id'] = row['sample_id']
        rule_features.append(feats)
        if idx % 500 == 0:
            print(f"Processed {idx} rows for rule-based features")
    rule_df = pd.DataFrame(rule_features)
    
    print("➡ Enriching with LLM features in batches...")
    llm_features_list = []
    for i in range(0, len(df), batch_size):
        batch_texts = df['catalog_content'].iloc[i:i+batch_size].tolist()
        batch_llm = enrich_features_batch(batch_texts)
        llm_features_list.extend(batch_llm)
        print(f"Processed batch {i} to {i+len(batch_texts)}")
    
    llm_df = pd.DataFrame(llm_features_list)
    
    print("➡ Merging features and one-hot encoding LLM outputs...")
    final_df = pd.concat([rule_df.reset_index(drop=True), llm_df.reset_index(drop=True)], axis=1)
    
    if 'llm_product_type' in final_df.columns:
        final_df = pd.get_dummies(final_df, columns=['llm_product_type'], prefix='product_type')
    
    if 'llm_flavors' in final_df.columns:
        all_flavors = set()
        for val in final_df['llm_flavors'].dropna():
            if isinstance(val, list):
                all_flavors.update(val)
        for f in all_flavors:
            final_df[f'flavor_{f.replace(" ", "_")}'] = final_df['llm_flavors'].apply(lambda x: int(f in x) if isinstance(x, list) else 0)
        final_df.drop(columns=['llm_flavors'], inplace=True)
    
    final_df.to_csv(output_csv, index=False)
    print(f"✅ ML-ready features saved to {output_csv}")

# ===============================
# Example usage
# ===============================
input_csv = "dataset/train.csv"
output_csv = "dataset/textoutput.csv"

process_dataset_optimized(input_csv, output_csv, batch_size=200)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.
Device set to use cpu


➡ Extracting rule-based features...
Processed 0 rows for rule-based features
Processed 500 rows for rule-based features
Processed 1000 rows for rule-based features
Processed 1500 rows for rule-based features
Processed 2000 rows for rule-based features
Processed 2500 rows for rule-based features
Processed 3000 rows for rule-based features
Processed 3500 rows for rule-based features
Processed 4000 rows for rule-based features
Processed 4500 rows for rule-based features
Processed 5000 rows for rule-based features
Processed 5500 rows for rule-based features
Processed 6000 rows for rule-based features
Processed 6500 rows for rule-based features
Processed 7000 rows for rule-based features
Processed 7500 rows for rule-based features
Processed 8000 rows for rule-based features
Processed 8500 rows for rule-based features
Processed 9000 rows for rule-based features
Processed 9500 rows for rule-based features
Processed 10000 rows for rule-based features
Processed 10500 rows for rule-based feature

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [ ]:
import pandas as pd
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
from time import sleep

# ===============================
# Loading Mistral 7B
# ===============================
print("➡ Loading Mistral 7B tokenizer...")
model_name = "mistralai/Mistral-7B-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("✅ Tokenizer loaded.")

print("➡ Loading Mistral 7B model (this may take several minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",       # automatically places model on GPU if available
    torch_dtype=torch.float16
)
print("✅ Model loaded.")

print("➡ Setting up text-generation pipeline...")
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)
print("✅ Pipeline ready.")

# ===============================
# Fast rule-based feature extractor
# ===============================
def extract_rule_based_features_fast(text):
    features = {}
    
    # Numeric
    value_match = re.search(r"Value:\s*([\d.]+)", text)
    features['weight_value'] = float(value_match.group(1)) if value_match else 0
    
    pack_match = re.search(r"Pack of (\d+)", text, re.IGNORECASE)
    features['pack_size'] = int(pack_match.group(1)) if pack_match else 1
    
    total_items_match = re.search(r"Includes .*? (\d+) (cookies|items|units)", text, re.IGNORECASE)
    features['total_items'] = int(total_items_match.group(1)) if total_items_match else features['pack_size']
    features['items_per_pack'] = features['total_items'] / (features['pack_size'] or 1)
    
    # Bullet points
    bullets = re.findall(r"Bullet Point \d+:\s*(.*)", text)
    features['num_bullets'] = len(bullets)
    
    all_text = text.replace("\n", " ").lower()
    features['description_length_words'] = len(all_text.split())
    features['description_length_chars'] = len(all_text)
    features['num_sentences'] = len(re.findall(r"[.!?]", all_text))
    features['num_numbers'] = len(re.findall(r"\d+", all_text))
    features['avg_word_length'] = sum(len(w) for w in all_text.split()) / (len(all_text.split()) or 1)
    
    # Boolean / categorical
    features['is_gluten_free'] = int("gluten-free" in all_text)
    features['is_nut_free'] = int("nut-free" in all_text)
    features['is_organic'] = int("organic" in all_text or "org" in all_text)
    features['is_wine'] = int("wine" in all_text or "alcohol" in all_text or "wine kit" in all_text)
    features['is_spice_seasoning'] = int("seasoning" in all_text or "spice" in all_text)
    features['is_liquid'] = int(re.search(r"(fl oz|l\b)", all_text) is not None)
    features['is_powder'] = int("powder" in all_text)
    features['contains_alcohol'] = features['is_wine']
    features['contains_spices'] = features['is_spice_seasoning']
    features['contains_ingredients'] = int("ingredients:" in all_text)
    
    # Flavors
    flavors = ["butter", "blue cheese", "basil", "cheddar", "vanilla"]
    for f in flavors:
        features[f'has_flavor_{f.replace(" ", "_")}'] = int(f in all_text)
    
    # Occasions
    occasions = ["birthday", "wedding", "anniversary", "party", "gift"]
    for occ in occasions:
        features[f'is_{occ}'] = int(occ in all_text)
    
    # Marketing keywords
    marketing_keywords = ["classic", "premium", "delicious", "trusted", "rich", "fun", "versatile"]
    features['num_marketing_keywords'] = sum(all_text.count(k) for k in marketing_keywords)
    
    # Derived
    features['num_occasions'] = sum(features[f'is_{occ}'] for occ in occasions)
    features['text_complexity'] = features['description_length_words'] / (features['num_sentences'] or 1)
    features['volume_per_unit'] = features['weight_value'] / (features['pack_size'] or 1)
    
    return features

# ===============================
# LLM enrichment (batched)
# ===============================
def enrich_features_batch(text_list):
    batch_results = []
    for i, text in enumerate(text_list):
        prompt = f"""Extract in JSON format:
- product_type (cookie, wine, spice, cereal, etc.)
- implicit giftable (1 or 0)
- flavor_profile (list)
Product Text: {text}"""
        try:
            output = generator(prompt, max_new_tokens=150)[0]['generated_text']
            json_start = output.find("{")
            json_end = output.rfind("}") + 1
            llm_features = eval(output[json_start:json_end])
        except Exception as e:
            llm_features = {
                "llm_product_type": None,
                "llm_is_giftable": None,
                "llm_flavors": None
            }
        batch_results.append(llm_features)
        if (i + 1) % 10 == 0:
            print(f"➡ LLM processed {i+1}/{len(text_list)} items in current batch")
        sleep(0.02)  # short pause to avoid overload
    return batch_results

# ===============================
# Full dataset processing
# ===============================
def process_dataset_optimized(input_csv, output_csv, batch_size=200):
    df = pd.read_csv(input_csv)
    if 'sample_id' not in df.columns:
        df['sample_id'] = df.index
    
    # Rule-based features
    print("➡ Extracting rule-based features...")
    rule_features = []
    for idx, row in df.iterrows():
        feats = extract_rule_based_features_fast(row['catalog_content'])
        feats['sample_id'] = row['sample_id']
        rule_features.append(feats)
        if idx % 500 == 0:
            print(f"Processed {idx}/{len(df)} rows for rule-based features")
    rule_df = pd.DataFrame(rule_features)
    
    # LLM enrichment
    print("➡ Enriching with LLM features in batches...")
    llm_features_list = []
    for i in range(0, len(df), batch_size):
        batch_texts = df['catalog_content'].iloc[i:i+batch_size].tolist()
        print(f"Processing LLM batch {i} to {i+len(batch_texts)}...")
        batch_llm = enrich_features_batch(batch_texts)
        llm_features_list.extend(batch_llm)
    llm_df = pd.DataFrame(llm_features_list)
    
    # Merge and one-hot encoding
    print("➡ Merging features and encoding LLM outputs...")
    final_df = pd.concat([rule_df.reset_index(drop=True), llm_df.reset_index(drop=True)], axis=1)
    
    if 'llm_product_type' in final_df.columns:
        final_df = pd.get_dummies(final_df, columns=['llm_product_type'], prefix='product_type')
    
    if 'llm_flavors' in final_df.columns:
        all_flavors = set()
        for val in final_df['llm_flavors'].dropna():
            if isinstance(val, list):
                all_flavors.update(val)
        for f in all_flavors:
            final_df[f'flavor_{f.replace(" ", "_")}'] = final_df['llm_flavors'].apply(lambda x: int(f in x) if isinstance(x, list) else 0)
        final_df.drop(columns=['llm_flavors'], inplace=True)
    
    final_df.to_csv(output_csv, index=False)
    print(f"✅ ML-ready features saved to {output_csv}")

# ===============================
# Run
# ===============================
input_csv = "dataset/train.csv"
output_csv = "dataset/textoutput.csv"

process_dataset_optimized(input_csv, output_csv, batch_size=200)


➡ Loading Mistral 7B tokenizer...
✅ Tokenizer loaded.
➡ Loading Mistral 7B model (this may take several minutes)...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.
Device set to use cpu


✅ Model loaded.
➡ Setting up text-generation pipeline...
✅ Pipeline ready.
➡ Extracting rule-based features...
Processed 0/75000 rows for rule-based features
Processed 500/75000 rows for rule-based features
Processed 1000/75000 rows for rule-based features
Processed 1500/75000 rows for rule-based features
Processed 2000/75000 rows for rule-based features
Processed 2500/75000 rows for rule-based features
Processed 3000/75000 rows for rule-based features
Processed 3500/75000 rows for rule-based features
Processed 4000/75000 rows for rule-based features
Processed 4500/75000 rows for rule-based features
Processed 5000/75000 rows for rule-based features
Processed 5500/75000 rows for rule-based features
Processed 6000/75000 rows for rule-based features
Processed 6500/75000 rows for rule-based features
Processed 7000/75000 rows for rule-based features
Processed 7500/75000 rows for rule-based features
Processed 8000/75000 rows for rule-based features
Processed 8500/75000 rows for rule-based fe

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
